In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
print(torch.__version__)

2.14.0+cpu


In [7]:
# Set a "seed" so the random numbers are the same every time you run this.
# Useful for reproducibility — you and I will see the exact same results.
torch.manual_seed(0)

# Create 200 fake data points, each with 4 features (just random numbers for now).
# Think of these 4 numbers as stand-ins for things like packet size, duration, etc.
X = torch.randn(200, 4)

# Create the "correct answers" (labels) for each of those 200 points.
# Our made-up rule: if the 4 features add up to more than 0, label it 1, otherwise 0.
# This gives the model something learnable to figure out.
# .float() converts True/False into 1.0/0.0 (numbers the model can work with).
# .unsqueeze(1) reshapes it from a flat list into a column — PyTorch expects labels
# in this [200, 1] shape to match the model's output shape later.
y = (X.sum(dim=1) > 0).float().unsqueeze(1)

# Print the shapes just to confirm: 200 rows of 4 features, 200 rows of 1 label each.
print(X.shape, y.shape)

torch.Size([200, 4]) torch.Size([200, 1])


In [9]:
class TinyNet(nn.Module):
    # This runs once, when you create the model — it defines the model's building blocks (layers).
    def __init__(self):
        super().__init__()  # required boilerplate — sets up PyTorch's internal machinery

        # First layer: takes in 4 numbers (our features) and outputs 8 numbers.
        # These 8 are the model's own "internal representation" — not meaningful to us directly.
        self.fc1 = nn.Linear(4, 8)

        # Second layer: takes those 8 numbers and boils them down to 1 final number.
        self.fc2 = nn.Linear(8, 1)

        # ReLU is an "activation function" — it adds non-linearity, letting the model
        # learn more than just straight lines/simple math. Common default choice.
        self.relu = nn.ReLU()

        # Sigmoid squashes the final output into a range between 0 and 1 —
        # perfect for "probability of being an attack" style predictions.
        self.sigmoid = nn.Sigmoid()

    # This runs every time you actually pass data through the model (a "forward pass").
    def forward(self, x):
        x = self.relu(self.fc1(x))   # data flows through layer 1, then ReLU
        x = self.sigmoid(self.fc2(x)) # then through layer 2, then Sigmoid
        return x  # final prediction, a number between 0 and 1

# Create an actual instance of the model
model = TinyNet()

# Print it out to see the structure you just defined
print(model)

TinyNet(
  (fc1): Linear(in_features=4, out_features=8, bias=True)
  (fc2): Linear(in_features=8, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)


In [11]:
# BCELoss = "Binary Cross-Entropy Loss" — the standard way to measure error
# for yes/no (binary) predictions like ours (attack vs. normal).
# It compares the model's predicted probability (0 to 1) against the true label (0 or 1).
criterion = nn.BCELoss()

# The optimizer is what actually adjusts the model's internal numbers (weights)
# to reduce the loss. Adam is a reliable, commonly-used default choice.
# lr = "learning rate" — how big a step it takes each update. 0.01 is a reasonable default.
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [13]:
# Loop through the whole dataset 50 times (50 "epochs")
for epoch in range(50):

    optimizer.zero_grad()        # Clear out gradients from the previous step —
                                  # PyTorch accumulates them by default, so this resets it.

    outputs = model(X)           # Forward pass: feed all 200 samples through the model,
                                  # get back 200 predictions.

    loss = criterion(outputs, y) # Compare predictions to true labels, get a single
                                  # number representing "how wrong" the model currently is.

    loss.backward()              # Backward pass: PyTorch automatically calculates how much
                                  # each weight in the model contributed to that error.

    optimizer.step()             # Use those calculations to actually nudge the weights
                                  # in the direction that reduces the error.

    # Every 10 epochs, print the current loss so we can watch it improve over time.
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 0.6545
Epoch 10, Loss: 0.5741
Epoch 20, Loss: 0.4698
Epoch 30, Loss: 0.3572
Epoch 40, Loss: 0.2572


In [15]:
# torch.no_grad() tells PyTorch "don't bother tracking gradients here" —
# we're just checking results now, not training, so this is faster and uses less memory.
with torch.no_grad():

    # Run all 200 samples through the trained model again.
    # If the model's output is > 0.5, we call it a prediction of "1", otherwise "0".
    preds = (model(X) > 0.5).float()

    # Compare predictions to the true labels, and average how often they match —
    # this gives us accuracy as a percentage.
    accuracy = (preds == y).float().mean()

    print(f"Accuracy: {accuracy.item():.2%}")

Accuracy: 96.50%
